# CONTRAFold
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
from pathlib import Path

In [ ]:
method_name = "CONTRAFold"
base = Path.cwd()

print(f"Base directory: {base}")

In [ ]:
os.makedirs('../tools', exist_ok=True)
%cd ../'tools'
!wget http://contra.stanford.edu/contrafold/contrafold_v2_02.tar.gz
!tar xzf contrafold_v2_02.tar.gz

In [ ]:
os.chdir("contrafold/src")

os.system('cat Makefile | awk \'{if($1=="CXXFLAGS"){print $0,"-fpermissive"}else{print $0}}\' > tmp')
os.system('mv tmp Makefile')
os.system('cat Utilities.hpp | awk \'{if($2=="<vector>"){print $0; print "#include <limits.h>"}else{print $0}}\' > tmp')
os.system('mv tmp Utilities.hpp')

!make clean
!make

# Return to methods directory
os.chdir("../../../methods")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
exe_path = base.parent / 'tools' / 'contrafold' / 'src' / 'contrafold'

def run_folding(fasta_name):
    out_file_name = "CONTRAFold_folded_seq.fasta"
    gamma = 0.9

    if os.path.exists("CONTRAFold_tmp"):
        os.remove("CONTRAFold_tmp")

    cmd = f"{exe_path} predict {fasta_name} --gamma {gamma}"
    result = os.system(f"{cmd} > CONTRAFold_tmp 2>/dev/null")

    if not os.path.exists("CONTRAFold_tmp"):
        return None

    with open(out_file_name, "w") as fout:
        if os.path.exists("CONTRAFold_tmp"):
            for line in open("CONTRAFold_tmp"):
                if ">structure" not in line:
                    fout.write(line)
            os.remove("CONTRAFold_tmp")
           
    return out_file_name

In [ ]:
os.makedirs('../prediction', exist_ok=True)
out_fasta_name = '../prediction/CONTRAFold.fasta'
if os.path.exists(out_fasta_name):
    os.remove(out_fasta_name)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")
for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    with open("CONTRAFold_tmp.fasta", "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    dot_file_name = run_folding("CONTRAFold_tmp.fasta")

    if dot_file_name and os.path.exists(dot_file_name):
        os.system("cat " + dot_file_name + " >> " + out_fasta_name)

    if os.path.exists("CONTRAFold_tmp.fasta"):
        os.remove("CONTRAFold_tmp.fasta")
    if dot_file_name and os.path.exists(dot_file_name):
        os.remove(dot_file_name)

    elapsed_time = time.time() - start_time
    print(f"{elapsed_time: .1f} s")